# LogiScan Defense-Prep Training Run — scripts embedded, data from input

The two TRAINING SCRIPTS are embedded in this notebook (no script uploads needed).
The DATA is uploaded by you: create a private dataset **`logiscan-data`** containing
the 2 JSON files (`stage1_splits.json` and `unified_training_data_v1.3.json` — at
the root or inside a `data/` subfolder; the cell below finds them anywhere under
`/kaggle/input`). Attach the dataset, select the **T4x2** accelerator, enable
**Internet**, run all cells.

Jobs:
1. Stage 1 gatekeeper retrain (DistilBERT, replaces the degenerate stage1_v12)
2. Stage 3 train/test split variation (seeds 42/2024/7) — answers the
   "train/test split variation" supervisor question with mean +/- std.

Notes:
- Both training loops use nn.DataParallel: the batch-size constants are
  PER-GPU; the loader batch auto-scales x2 on T4x2.
- Stage 1 saves best-val-F1 checkpoint; tune `EPOCHS_STAGE1` / `SEEDS_STAGE3`
  in the data cell if desired.
- Expected wall time on T4x2: stage 1 ~10-20 min, 3x stage 3 ~1.5-2.5 h.
  Kaggle sessions allow 12 h — let it run.

After the run, download (from the last cell):
- `models/stage1_v13_classifier.zip` -> unzip into repo `models/`, set
  `STAGE1_MODEL_PATH=./models/stage1_v13_classifier` in `.env`
- `cloud_training/results/seed_variation.json` -> paste mean +/- std into
  `docs/slides/09-validation-results.md` (two placeholder cells)


## 0. Environment setup


In [ ]:
!pip install -q transformers scikit-learn tqdm accelerate onnxscript pandas numpy
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())
print('GPUs visible:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f'  [{i}]', torch.cuda.get_device_name(i))
assert torch.cuda.device_count() >= 1, 'no GPU allocated - set accelerator to T4x2'


## 1. Data from input + training knobs


In [ ]:
import shutil
from pathlib import Path

# --- locate the uploaded data (attached as the 'logiscan-data' dataset) ---
TARGETS = ["stage1_splits.json", "unified_training_data_v1.3.json"]
Path("data").mkdir(exist_ok=True)
missing = []
for name in TARGETS:
    dest = Path("data") / name
    if dest.exists():
        print(f"found local: data/{name}")
        continue
    hits = list(Path("/kaggle/input").rglob(name))
    if hits:
        shutil.copy(hits[0], dest)
        print(f"copied from input: {hits[0]} -> data/{name}")
    else:
        missing.append(name)
if missing:
    raise SystemExit(
        "!! missing data files: "
        + ", ".join(missing)
        + "\nattach the 'logiscan-data' dataset (2 JSONs) or put them in ./data"
    )

# --- single place to tweak training knobs ---
EPOCHS_STAGE1 = 6          # stage 1 gatekeeper epochs (best-val-F1 checkpointing)
SEEDS_STAGE3 = [42, 2024, 7]
BATCH_STAGE1 = 32          # per-GPU (loader auto x2 on T4x2)
BATCH_STAGE3 = 16          # per-GPU (loader auto x2 on T4x2)


## 2. Stage 1 gatekeeper retrain


In [ ]:
import argparse, base64

_STAGE1_SRC = base64.b64decode("IiIiClRyYWluIFN0YWdlIDE6IEJpbmFyeSBBcmd1bWVudCBHYXRlIChEaXN0aWxCRVJUKSBvbiB0aGUgc3RhZ2UxX3NwbGl0cyBkYXRhc2V0LgoKS2FnZ2xlLXJlYWR5IHJlcGxhY2VtZW50IGZvciB0aGUgZGVnZW5lcmF0ZSBgbW9kZWxzL3N0YWdlMV92MTJgIGNoZWNrcG9pbnQKKHdoaWNoIG91dHB1dHMgc2FsaWVuY2U9MS4wMDAgZm9yIGV2ZXJ5IGlucHV0KS4gS2V5IHByb3BlcnRpZXM6CgotIFJlYWRzIGBkYXRhL3N0YWdlMV9zcGxpdHMuanNvbmAgKGZpeGVkIHRyYWluL3ZhbC90ZXN0OiAxMCw1ODQgLyAxLDE1NSAvIDEsMTU1KS4KLSBCYXNlIG1vZGVsIGBkaXN0aWxiZXJ0LWJhc2UtdW5jYXNlZGAsIDIgbGFiZWxzOiAwID0gbm9uLWFyZ3VtZW50LCAxID0gYXJndW1lbnQuCiAgVGhlIGJhY2tlbmQgbWFwcyBgc29mdG1heChsb2dpdHMpWzFdYCAtPiBzYWxpZW5jZSAoc3RhZ2UxX2dhdGVrZWVwZXIucHkpLCBzbwogIGNsYXNzIGluZGV4IDEgTVVTVCBiZSAiYXJndW1lbnQiIOKAlCB0aGlzIHNjcmlwdCBjb25maWd1cmVzIGlkMmxhYmVsIGFjY29yZGluZ2x5LgotIEJlc3QtdmFsLW1hY3JvLUYxIGNoZWNrcG9pbnRpbmcsIGxpbmVhciB3YXJtdXAgKDEwJSksIEFkYW1XLCBncmFkIGNsaXBwaW5nLgotIEV4cG9ydHMgT05OWCAobm9uLWZhdGFsIG9uIGZhaWx1cmUpIHdpdGggZHluYW1pYyBzZXF1ZW5jZSBsZW5ndGguCgpVc2FnZToKICAgIHB5dGhvbiBjbG91ZF90cmFpbmluZy9zY3JpcHRzL3RyYWluX3N0YWdlMV92MTMucHkgXAogICAgICAgIC0tZGF0YSBkYXRhL3N0YWdlMV9zcGxpdHMuanNvbiBcCiAgICAgICAgLS1vdXRwdXQtZGlyIG1vZGVscy9zdGFnZTFfdjEzX2NsYXNzaWZpZXIgXAogICAgICAgIC0tZXBvY2hzIDMgLS1iYXRjaC1zaXplIDMyIC0tbHIgM2UtNQoiIiIKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBqc29uCmltcG9ydCBsb2dnaW5nCmltcG9ydCBzaHV0aWwKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHRvcmNoCmltcG9ydCB0b3JjaC5ubiBhcyBubgpmcm9tIHNrbGVhcm4ubWV0cmljcyBpbXBvcnQgYWNjdXJhY3lfc2NvcmUsIGNsYXNzaWZpY2F0aW9uX3JlcG9ydCwgZjFfc2NvcmUKZnJvbSB0b3JjaC51dGlscy5kYXRhIGltcG9ydCBEYXRhTG9hZGVyLCBEYXRhc2V0CmZyb20gdHFkbSBpbXBvcnQgdHFkbQpmcm9tIHRyYW5zZm9ybWVycyBpbXBvcnQgKAogICAgQXV0b0NvbmZpZywKICAgIEF1dG9Nb2RlbEZvclNlcXVlbmNlQ2xhc3NpZmljYXRpb24sCiAgICBBdXRvVG9rZW5pemVyLAogICAgZ2V0X2xpbmVhcl9zY2hlZHVsZV93aXRoX3dhcm11cCwKKQoKbG9nZ2luZy5iYXNpY0NvbmZpZyhsZXZlbD1sb2dnaW5nLklORk8pCmxvZ2dlciA9IGxvZ2dpbmcuZ2V0TG9nZ2VyKF9fbmFtZV9fKQoKREVWSUNFID0gdG9yY2guZGV2aWNlKCJjdWRhIiBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpIGVsc2UgImNwdSIpCkxBQkVMX05BTUVTID0gezA6ICJub25fYXJndW1lbnQiLCAxOiAiYXJndW1lbnQifQoKCmNsYXNzIEFyZ3VtZW50RGF0YXNldChEYXRhc2V0KToKICAgIGRlZiBfX2luaXRfXyhzZWxmLCB0ZXh0cywgbGFiZWxzLCB0b2tlbml6ZXIsIG1heF9sZW5ndGg9NTEyKToKICAgICAgICBzZWxmLnRleHRzID0gdGV4dHMKICAgICAgICBzZWxmLmxhYmVscyA9IGxhYmVscwogICAgICAgIHNlbGYudG9rZW5pemVyID0gdG9rZW5pemVyCiAgICAgICAgc2VsZi5tYXhfbGVuZ3RoID0gbWF4X2xlbmd0aAoKICAgIGRlZiBfX2xlbl9fKHNlbGYpOgogICAgICAgIHJldHVybiBsZW4oc2VsZi50ZXh0cykKCiAgICBkZWYgX19nZXRpdGVtX18oc2VsZiwgaWR4KToKICAgICAgICBlbmMgPSBzZWxmLnRva2VuaXplcigKICAgICAgICAgICAgc2VsZi50ZXh0c1tpZHhdLAogICAgICAgICAgICB0cnVuY2F0aW9uPVRydWUsCiAgICAgICAgICAgIHBhZGRpbmc9Im1heF9sZW5ndGgiLAogICAgICAgICAgICBtYXhfbGVuZ3RoPXNlbGYubWF4X2xlbmd0aCwKICAgICAgICAgICAgcmV0dXJuX3RlbnNvcnM9InB0IiwKICAgICAgICApCiAgICAgICAgcmV0dXJuIHsKICAgICAgICAgICAgImlucHV0X2lkcyI6IGVuY1siaW5wdXRfaWRzIl0uc3F1ZWV6ZSgwKSwKICAgICAgICAgICAgImF0dGVudGlvbl9tYXNrIjogZW5jWyJhdHRlbnRpb25fbWFzayJdLnNxdWVlemUoMCksCiAgICAgICAgICAgICJsYWJlbHMiOiB0b3JjaC50ZW5zb3Ioc2VsZi5sYWJlbHNbaWR4XSwgZHR5cGU9dG9yY2gubG9uZyksCiAgICAgICAgfQoKCmRlZiBsb2FkX2RhdGEocGF0aDogUGF0aCk6CiAgICB3aXRoIG9wZW4ocGF0aCkgYXMgZjoKICAgICAgICBzcGxpdHMgPSBqc29uLmxvYWQoZikKICAgIG91dCA9IHt9CiAgICBmb3Iga2V5IGluICgidHJhaW4iLCAidmFsIiwgInRlc3QiKToKICAgICAgICByZWNvcmRzID0gc3BsaXRzW2tleV0KICAgICAgICBvdXRba2V5XSA9IChbZFsidGV4dCJdIGZvciBkIGluIHJlY29yZHNdLCBbZFsibGFiZWwiXSBmb3IgZCBpbiByZWNvcmRzXSkKICAgIHJldHVybiBvdXQKCgpkZWYgZXZhbHVhdGUobW9kZWwsIGxvYWRlcikgLT4gZGljdDoKICAgIG1vZGVsLmV2YWwoKQogICAgYWxsX3ByZWRzLCBhbGxfbGFiZWxzID0gW10sIFtdCiAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICBmb3IgYmF0Y2ggaW4gbG9hZGVyOgogICAgICAgICAgICBvdXRwdXRzID0gbW9kZWwoCiAgICAgICAgICAgICAgICBiYXRjaFsiaW5wdXRfaWRzIl0udG8oREVWSUNFKSwgYXR0ZW50aW9uX21hc2s9YmF0Y2hbImF0dGVudGlvbl9tYXNrIl0udG8oREVWSUNFKQogICAgICAgICAgICApCiAgICAgICAgICAgIHByZWRzID0gdG9yY2guYXJnbWF4KG91dHB1dHMubG9naXRzLCBkaW09MSkKICAgICAgICAgICAgYWxsX3ByZWRzLmV4dGVuZChwcmVkcy5jcHUoKS50b2xpc3QoKSkKICAgICAgICAgICAgYWxsX2xhYmVscy5leHRlbmQoYmF0Y2hbImxhYmVscyJdLnRvbGlzdCgpKQogICAgcmV0dXJuIHsKICAgICAgICAiYWNjdXJhY3kiOiBhY2N1cmFjeV9zY29yZShhbGxfbGFiZWxzLCBhbGxfcHJlZHMpLAogICAgICAgICJtYWNyb19mMSI6IGYxX3Njb3JlKGFsbF9sYWJlbHMsIGFsbF9wcmVkcywgYXZlcmFnZT0ibWFjcm8iLCB6ZXJvX2RpdmlzaW9uPTApLAogICAgICAgICJwcmVkcyI6IGFsbF9wcmVkcywKICAgICAgICAibGFiZWxzIjogYWxsX2xhYmVscywKICAgIH0KCgpkZWYgZXhwb3J0X29ubngobW9kZWwsIHRva2VuaXplciwgb3V0cHV0X2Rpcik6CiAgICBsb2dnZXIuaW5mbygiRXhwb3J0aW5nIHRvIE9OTlguLi4iKQogICAgbW9kZWwuZXZhbCgpCiAgICBkdW1teSA9IHRva2VuaXplcigKICAgICAgICAidGVzdCBpbnB1dCIsIHJldHVybl90ZW5zb3JzPSJwdCIsIHBhZGRpbmc9Im1heF9sZW5ndGgiLCB0cnVuY2F0aW9uPVRydWUsIG1heF9sZW5ndGg9MjU2CiAgICApCiAgICBkdW1teV9pZHMgPSBkdW1teVsiaW5wdXRfaWRzIl0udG8oREVWSUNFKQogICAgZHVtbXlfbWFzayA9IGR1bW15WyJhdHRlbnRpb25fbWFzayJdLnRvKERFVklDRSkKCiAgICBvbm54X3BhdGggPSBvdXRwdXRfZGlyIC8gIm1vZGVsLm9ubngiCiAgICB0b3JjaC5vbm54LmV4cG9ydCgKICAgICAgICBtb2RlbCwKICAgICAgICAoZHVtbXlfaWRzLCBkdW1teV9tYXNrKSwKICAgICAgICBvbm54X3BhdGgsCiAgICAgICAgb3BzZXRfdmVyc2lvbj0xNCwKICAgICAgICBpbnB1dF9uYW1lcz1bImlucHV0X2lkcyIsICJhdHRlbnRpb25fbWFzayJdLAogICAgICAgIG91dHB1dF9uYW1lcz1bImxvZ2l0cyJdLAogICAgICAgIGR5bmFtaWNfYXhlcz17CiAgICAgICAgICAgICJpbnB1dF9pZHMiOiB7MDogImJhdGNoIiwgMTogInNlcXVlbmNlIn0sCiAgICAgICAgICAgICJhdHRlbnRpb25fbWFzayI6IHswOiAiYmF0Y2giLCAxOiAic2VxdWVuY2UifSwKICAgICAgICAgICAgImxvZ2l0cyI6IHswOiAiYmF0Y2gifSwKICAgICAgICB9LAogICAgKQogICAgbG9nZ2VyLmluZm8oZiJPTk5YIGV4cG9ydCBjb21wbGV0ZToge29ubnhfcGF0aC5uYW1lfSAoe29ubnhfcGF0aC5zdGF0KCkuc3Rfc2l6ZSAvICgxMDI0KioyKTouMWZ9IE1CKSIpCiAgICB3aXRoIG9wZW4ob3V0cHV0X2RpciAvICJtb2RlbF9jb25maWdfb25ueC5qc29uIiwgInciKSBhcyBmOgogICAgICAgIGpzb24uZHVtcCh7ImxhYmVscyI6IExBQkVMX05BTUVTLCAibnVtX2xhYmVscyI6IDJ9LCBmLCBpbmRlbnQ9MikKICAgIHppcF9wYXRoID0gc2h1dGlsLm1ha2VfYXJjaGl2ZShzdHIob3V0cHV0X2Rpci5wYXJlbnQgLyBvdXRwdXRfZGlyLm5hbWUpLCAiemlwIiwgb3V0cHV0X2RpcikKICAgIGxvZ2dlci5pbmZvKGYiUGFja2FnZSByZWFkeToge3ppcF9wYXRofSIpCgoKZGVmIHRyYWluKGFyZ3MpOgogICAgbG9nZ2VyLmluZm8oZiJVc2luZyBkZXZpY2U6IHtERVZJQ0V9ICh7dG9yY2guY3VkYS5nZXRfZGV2aWNlX25hbWUoMCkgaWYgREVWSUNFLnR5cGUgPT0gJ2N1ZGEnIGVsc2UgJ2NwdSd9KSIpCiAgICBucC5yYW5kb20uc2VlZChhcmdzLnNlZWQpCiAgICB0b3JjaC5tYW51YWxfc2VlZChhcmdzLnNlZWQpCgogICAgZGF0YV9wYXRoID0gUGF0aChhcmdzLmRhdGEpCiAgICBpZiBub3QgZGF0YV9wYXRoLmV4aXN0cygpOgogICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKGYiRGF0YSBub3QgZm91bmQgYXQge2RhdGFfcGF0aH0iKQogICAgb3V0cHV0X2RpciA9IFBhdGgoYXJncy5vdXRwdXRfZGlyKQogICAgb3V0cHV0X2Rpci5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCgogICAgZGF0YSA9IGxvYWRfZGF0YShkYXRhX3BhdGgpCiAgICBYX3RyYWluLCB5X3RyYWluID0gZGF0YVsidHJhaW4iXQogICAgWF92YWwsIHlfdmFsID0gZGF0YVsidmFsIl0KICAgIFhfdGVzdCwgeV90ZXN0ID0gZGF0YVsidGVzdCJdCiAgICBsb2dnZXIuaW5mbygKICAgICAgICBmIlRyYWluOiB7bGVuKFhfdHJhaW4pfSAvIFZhbDoge2xlbihYX3ZhbCl9IC8gVGVzdDoge2xlbihYX3Rlc3QpfSAiCiAgICAgICAgZiIocG9zaXRpdmVzOiB7c3VtKHlfdHJhaW4pfSkiCiAgICApCgogICAgdG9rZW5pemVyID0gQXV0b1Rva2VuaXplci5mcm9tX3ByZXRyYWluZWQoYXJncy5iYXNlX21vZGVsKQogICAgdHJhaW5fZHMgPSBBcmd1bWVudERhdGFzZXQoWF90cmFpbiwgeV90cmFpbiwgdG9rZW5pemVyLCBtYXhfbGVuZ3RoPWFyZ3MubWF4X2xlbmd0aCkKICAgIHZhbF9kcyA9IEFyZ3VtZW50RGF0YXNldChYX3ZhbCwgeV92YWwsIHRva2VuaXplciwgbWF4X2xlbmd0aD1hcmdzLm1heF9sZW5ndGgpCiAgICB0ZXN0X2RzID0gQXJndW1lbnREYXRhc2V0KFhfdGVzdCwgeV90ZXN0LCB0b2tlbml6ZXIsIG1heF9sZW5ndGg9YXJncy5tYXhfbGVuZ3RoKQogICAgbl9ncHVzID0gdG9yY2guY3VkYS5kZXZpY2VfY291bnQoKSBpZiBERVZJQ0UudHlwZSA9PSAiY3VkYSIgZWxzZSAwCiAgICBsb2FkZXJfYmF0Y2ggPSBhcmdzLmJhdGNoX3NpemUgKiBuX2dwdXMgaWYgbl9ncHVzID4gMSBlbHNlIGFyZ3MuYmF0Y2hfc2l6ZQogICAgdHJhaW5fbG9hZGVyID0gRGF0YUxvYWRlcih0cmFpbl9kcywgYmF0Y2hfc2l6ZT1sb2FkZXJfYmF0Y2gsIHNodWZmbGU9VHJ1ZSkKICAgIHZhbF9sb2FkZXIgPSBEYXRhTG9hZGVyKHZhbF9kcywgYmF0Y2hfc2l6ZT1sb2FkZXJfYmF0Y2ggKiAyKQogICAgdGVzdF9sb2FkZXIgPSBEYXRhTG9hZGVyKHRlc3RfZHMsIGJhdGNoX3NpemU9bG9hZGVyX2JhdGNoICogMikKCiAgICBjb25maWcgPSBBdXRvQ29uZmlnLmZyb21fcHJldHJhaW5lZChhcmdzLmJhc2VfbW9kZWwpCiAgICBjb25maWcubnVtX2xhYmVscyA9IDIKICAgIGNvbmZpZy5pZDJsYWJlbCA9IHtzdHIoayk6IHYgZm9yIGssIHYgaW4gTEFCRUxfTkFNRVMuaXRlbXMoKX0KICAgIGNvbmZpZy5sYWJlbDJpZCA9IHt2OiBrIGZvciBrLCB2IGluIExBQkVMX05BTUVTLml0ZW1zKCl9CgogICAgbW9kZWwgPSBBdXRvTW9kZWxGb3JTZXF1ZW5jZUNsYXNzaWZpY2F0aW9uLmZyb21fcHJldHJhaW5lZCgKICAgICAgICBhcmdzLmJhc2VfbW9kZWwsIGNvbmZpZz1jb25maWcKICAgICkudG8oREVWSUNFKQogICAgbG9nZ2VyLmluZm8oZiJNb2RlbCBwYXJhbXM6IHtzdW0ocC5udW1lbCgpIGZvciBwIGluIG1vZGVsLnBhcmFtZXRlcnMoKSk6LH0iKQoKICAgIHVzZV9kcCA9IG5fZ3B1cyA+IDEKICAgIGlmIHVzZV9kcDoKICAgICAgICB0b3JjaC5iYWNrZW5kcy5jdWRubi5iZW5jaG1hcmsgPSBUcnVlCiAgICAgICAgbW9kZWwgPSBubi5EYXRhUGFyYWxsZWwobW9kZWwpCiAgICAgICAgbG9nZ2VyLmluZm8oCiAgICAgICAgICAgIGYiRGF0YVBhcmFsbGVsIGFjdGl2ZSBhY3Jvc3Mge25fZ3B1c30gR1BVczsgZWZmZWN0aXZlIHBlci1HUFUgYmF0Y2ggPSB7YXJncy5iYXRjaF9zaXplfSIKICAgICAgICApCgogICAgb3B0aW1pemVyID0gdG9yY2gub3B0aW0uQWRhbVcobW9kZWwucGFyYW1ldGVycygpLCBscj1hcmdzLmxyLCB3ZWlnaHRfZGVjYXk9MC4wMSkKICAgIHRvdGFsX3N0ZXBzID0gbGVuKHRyYWluX2xvYWRlcikgKiBhcmdzLmVwb2NocwogICAgc2NoZWR1bGVyID0gZ2V0X2xpbmVhcl9zY2hlZHVsZV93aXRoX3dhcm11cCgKICAgICAgICBvcHRpbWl6ZXIsIG51bV93YXJtdXBfc3RlcHM9dG90YWxfc3RlcHMgLy8gMTAsIG51bV90cmFpbmluZ19zdGVwcz10b3RhbF9zdGVwcwogICAgKQogICAgc2NhbGVyID0gdG9yY2guYW1wLkdyYWRTY2FsZXIoImN1ZGEiKSBpZiBERVZJQ0UudHlwZSA9PSAiY3VkYSIgZWxzZSBOb25lCiAgICBsb3NzX2ZuID0gbm4uQ3Jvc3NFbnRyb3B5TG9zcygpCgogICAgYmVzdF9mMSA9IDAuMAogICAgZm9yIGVwb2NoIGluIHJhbmdlKGFyZ3MuZXBvY2hzKToKICAgICAgICBtb2RlbC50cmFpbigpCiAgICAgICAgdHJhaW5fbG9zcyA9IDAuMAogICAgICAgIHBiYXIgPSB0cWRtKHRyYWluX2xvYWRlciwgZGVzYz1mIkVwb2NoIHtlcG9jaCArIDF9L3thcmdzLmVwb2Noc30iKQogICAgICAgIGZvciBiYXRjaCBpbiBwYmFyOgogICAgICAgICAgICBvcHRpbWl6ZXIuemVyb19ncmFkKCkKICAgICAgICAgICAgaW5wdXRfaWRzID0gYmF0Y2hbImlucHV0X2lkcyJdLnRvKERFVklDRSkKICAgICAgICAgICAgYXR0ZW50aW9uX21hc2sgPSBiYXRjaFsiYXR0ZW50aW9uX21hc2siXS50byhERVZJQ0UpCiAgICAgICAgICAgIGxhYmVsc19iYXRjaCA9IGJhdGNoWyJsYWJlbHMiXS50byhERVZJQ0UpCgogICAgICAgICAgICBpZiBzY2FsZXI6CiAgICAgICAgICAgICAgICB3aXRoIHRvcmNoLmFtcC5hdXRvY2FzdChkZXZpY2VfdHlwZT0iY3VkYSIpOgogICAgICAgICAgICAgICAgICAgIG91dHB1dHMgPSBtb2RlbChpbnB1dF9pZHMsIGF0dGVudGlvbl9tYXNrPWF0dGVudGlvbl9tYXNrKQogICAgICAgICAgICAgICAgICAgIGxvc3MgPSBsb3NzX2ZuKG91dHB1dHMubG9naXRzLCBsYWJlbHNfYmF0Y2gpCiAgICAgICAgICAgICAgICBzY2FsZXIuc2NhbGUobG9zcykuYmFja3dhcmQoKQogICAgICAgICAgICAgICAgc2NhbGVyLnVuc2NhbGVfKG9wdGltaXplcikKICAgICAgICAgICAgICAgIHRvcmNoLm5uLnV0aWxzLmNsaXBfZ3JhZF9ub3JtXyhtb2RlbC5wYXJhbWV0ZXJzKCksIDEuMCkKICAgICAgICAgICAgICAgIHNjYWxlci5zdGVwKG9wdGltaXplcikKICAgICAgICAgICAgICAgIHNjYWxlci51cGRhdGUoKQogICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgb3V0cHV0cyA9IG1vZGVsKGlucHV0X2lkcywgYXR0ZW50aW9uX21hc2s9YXR0ZW50aW9uX21hc2spCiAgICAgICAgICAgICAgICBsb3NzID0gbG9zc19mbihvdXRwdXRzLmxvZ2l0cywgbGFiZWxzX2JhdGNoKQogICAgICAgICAgICAgICAgbG9zcy5iYWNrd2FyZCgpCiAgICAgICAgICAgICAgICB0b3JjaC5ubi51dGlscy5jbGlwX2dyYWRfbm9ybV8obW9kZWwucGFyYW1ldGVycygpLCAxLjApCiAgICAgICAgICAgICAgICBvcHRpbWl6ZXIuc3RlcCgpCgogICAgICAgICAgICBzY2hlZHVsZXIuc3RlcCgpCiAgICAgICAgICAgIHRyYWluX2xvc3MgKz0gbG9zcy5pdGVtKCkKICAgICAgICAgICAgcGJhci5zZXRfcG9zdGZpeCh7Imxvc3MiOiBmIntsb3NzLml0ZW0oKTouNGZ9In0pCgogICAgICAgIHZhbF9tZXRyaWNzID0gZXZhbHVhdGUobW9kZWwsIHZhbF9sb2FkZXIpCiAgICAgICAgZjEgPSB2YWxfbWV0cmljc1sibWFjcm9fZjEiXQogICAgICAgIGxvZ2dlci5pbmZvKAogICAgICAgICAgICBmIkVwb2NoIHtlcG9jaCArIDF9OiBsb3NzPXt0cmFpbl9sb3NzIC8gbGVuKHRyYWluX2xvYWRlcik6LjRmfSwgIgogICAgICAgICAgICBmInZhbF9hY2M9e3ZhbF9tZXRyaWNzWydhY2N1cmFjeSddOi40Zn0sIHZhbF9tYWNyb19mMT17ZjE6LjRmfSIKICAgICAgICApCiAgICAgICAgaWYgZjEgPiBiZXN0X2YxOgogICAgICAgICAgICBiZXN0X2YxID0gZjEKICAgICAgICAgICAgbG9nZ2VyLmluZm8oZiJOZXcgYmVzdCBtYWNybyBGMToge2YxOi40Zn0uIFNhdmluZyBjaGVja3BvaW50Li4uIikKICAgICAgICAgICAgc2F2ZV9tb2RlbCA9IG1vZGVsLm1vZHVsZSBpZiBpc2luc3RhbmNlKG1vZGVsLCBubi5EYXRhUGFyYWxsZWwpIGVsc2UgbW9kZWwKICAgICAgICAgICAgc2F2ZV9tb2RlbC5zYXZlX3ByZXRyYWluZWQob3V0cHV0X2RpcikKICAgICAgICAgICAgdG9rZW5pemVyLnNhdmVfcHJldHJhaW5lZChvdXRwdXRfZGlyKQogICAgICAgICAgICB3aXRoIG9wZW4ob3V0cHV0X2RpciAvICJsYWJlbHMuanNvbiIsICJ3IikgYXMgZjoKICAgICAgICAgICAgICAgIGpzb24uZHVtcChMQUJFTF9OQU1FUywgZiwgaW5kZW50PTIpCgogICAgbG9nZ2VyLmluZm8oIj09PSBGaW5hbCBldmFsdWF0aW9uIChiZXN0IGNoZWNrcG9pbnQpID09PSIpCiAgICBiZXN0X21vZGVsID0gQXV0b01vZGVsRm9yU2VxdWVuY2VDbGFzc2lmaWNhdGlvbi5mcm9tX3ByZXRyYWluZWQoc3RyKG91dHB1dF9kaXIpKS50byhERVZJQ0UpCiAgICBmb3IgbmFtZSwgbG9hZGVyLCBsYWJlbHMgaW4gKAogICAgICAgICgidHJhaW4iLCB0cmFpbl9sb2FkZXIsIHlfdHJhaW4pLAogICAgICAgICgidmFsIiwgdmFsX2xvYWRlciwgeV92YWwpLAogICAgICAgICgidGVzdCIsIHRlc3RfbG9hZGVyLCB5X3Rlc3QpLAogICAgKToKICAgICAgICBtID0gZXZhbHVhdGUoYmVzdF9tb2RlbCwgbG9hZGVyKQogICAgICAgIHJlcG9ydCA9IGNsYXNzaWZpY2F0aW9uX3JlcG9ydCgKICAgICAgICAgICAgbVsibGFiZWxzIl0sIG1bInByZWRzIl0sIHRhcmdldF9uYW1lcz1saXN0KExBQkVMX05BTUVTLnZhbHVlcygpKSwgemVyb19kaXZpc2lvbj0wLCBkaWdpdHM9MwogICAgICAgICkKICAgICAgICBsb2dnZXIuaW5mbyhmIi0tLSB7bmFtZX0gKG49e2xlbihsYWJlbHMpfSkgLS0tXG57cmVwb3J0fSIpCgogICAgaWYgYXJncy5leHBvcnRfb25ueDoKICAgICAgICB0cnk6CiAgICAgICAgICAgIGV4cG9ydF9vbm54KGJlc3RfbW9kZWwsIHRva2VuaXplciwgb3V0cHV0X2RpcikKICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6ICAjIHByYWdtYTogbm8gY292ZXIgLSBleHBvcnQgaXMgYmVzdC1lZmZvcnQKICAgICAgICAgICAgbG9nZ2VyLndhcm5pbmcoZiJPTk5YIGV4cG9ydCBmYWlsZWQgKG5vbi1mYXRhbCk6IHtlfSIpCgogICAgbG9nZ2VyLmluZm8oZiJUcmFpbmluZyBjb21wbGV0ZS4gQmVzdCB2YWwgbWFjcm8gRjE6IHtiZXN0X2YxOi40Zn0iKQogICAgcmV0dXJuIGZsb2F0KGJlc3RfZjEpCg==").decode()
_NS1 = {}
exec(_STAGE1_SRC, _NS1)
print("stage 1 code loaded:", _NS1["train"].__name__)


In [ ]:
args = argparse.Namespace(
    data="data/stage1_splits.json",
    output_dir="models/stage1_v13_classifier",
    base_model="distilbert-base-uncased",
    epochs=EPOCHS_STAGE1,
    batch_size=BATCH_STAGE1,
    lr=3e-5,
    seed=42,
    max_length=512,
    export_onnx=False,
)
_NS1["train"](args)


In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

m = AutoModelForSequenceClassification.from_pretrained("models/stage1_v13_classifier").to("cuda")
t = AutoTokenizer.from_pretrained("models/stage1_v13_classifier")
for s in ["Therefore, all humans are mortal.", "Click the blue button to submit.", "Physics is the fundamental natural science that studies matter."]:
    inp = t(s, return_tensors="pt", truncation=True, max_length=512).to("cuda")
    with torch.no_grad():
        p = torch.softmax(m(**inp).logits, dim=-1)[0][1].item()
    print(f"salience={p:.3f}  |  {s[:50]}")


## 3. Stage 3 seed variation (3 runs)


In [ ]:
import argparse, base64

_STAGE3_SRC = base64.b64decode("IiIiClRyYWluIFN0YWdlIDM6IDI5LUNsYXNzIEZpbmUtR3JhaW5lZCBGYWxsYWN5IENsYXNzaWZpZXIgb24gdGhlIHYxLjMgZGF0YXNldC4KClJ1bnMgb24gS2FnZ2xlIChUNC9QMTAwIEdQVSkgb3IgbG9jYWxseS4gS2V5IGRpZmZlcmVuY2VzIHZzIHRoZSBsZWdhY3kKdHJhaW5fc3RhZ2UzLnB5IC8gdHJhaW5faGFyZGVuZWRfc3RhZ2UzLnB5OgoKLSBSZWFkcyBgZGF0YS91bmlmaWVkX3RyYWluaW5nX2RhdGFfdjEuMy5qc29uYCAoMTcsOTM4IHNhbXBsZXMpLgotIEZpbHRlcnMgb3V0IHRoZSA5OTYgbmVhci1taXNzIGhhcmQtbmVnYXRpdmUgc2FtcGxlcyAoc2NoZW1hIGxhY2tzIHRoZQogIGBmYWxsYWN5YCBrZXk7IHRob3NlIGJlbG9uZyB0byB0aGUgU3RhZ2UgMSBnYXRla2VlcGVyLCBub3QgdGhlIFN0YWdlIDMKICBmaW5lIGhlYWQpIC0+IDE2LDk0MiBsYWJlbGVkIHNhbXBsZXMgYWNyb3NzIDI5IGNsYXNzZXMuCi0gQmFzZSBtb2RlbCBtYXRjaGVzIHByb2R1Y3Rpb24gKGBtaWNyb3NvZnQvZGViZXJ0YS12My1zbWFsbGAsIHNpbmdsZS1oZWFkLAogIHNhbWUgYXJjaGl0ZWN0dXJlIGFzIGBtb2RlbHMvcGhhc2U0X2ZpbmFsX21vZGVsYCkuCi0gc3FydC1pbnZlcnNlLWZyZXF1ZW5jeSBjbGFzcyB3ZWlnaHRpbmcgKDEvc3FydChjb3VudCksIHJlbm9ybWFsaXplZCkuCi0gUGVyLWNsYXNzIGNsYXNzaWZpY2F0aW9uIHJlcG9ydCB3aXRoIGZvcm1hbCBjbGFzc2VzIGhpZ2hsaWdodGVkIHNvIHRoZQogIFBoYXNlIDggIzUvIzggdmFsaWRhdGlvbiBpcyBhIG9uZS1nbGFuY2UgY2hlY2suCi0gTXVsdGktR1BVIHZpYSBgbm4uRGF0YVBhcmFsbGVsYCB3aGVuID4xIENVREEgZGV2aWNlIGlzIHZpc2libGU7IHRoZSBsb2FkZXIKICBiYXRjaCBpcyBhdXRvLXNjYWxlZCAoYmF0Y2hfc2l6ZSB4IG5fZ3B1cykgc28gdGhlIGVmZmVjdGl2ZSBiYXRjaCBwZXIgR1BVCiAgc3RheXMgYXQgYC0tYmF0Y2gtc2l6ZWAuCgpVc2FnZToKICAgIHB5dGhvbiBjbG91ZF90cmFpbmluZy9zY3JpcHRzL3RyYWluX3N0YWdlM192MTMucHkgLS1kYXRhIGRhdGEvdW5pZmllZF90cmFpbmluZ19kYXRhX3YxLjMuanNvbgogICAgIyBLYWdnbGUgVDR4MiAoMTYvR1BVKTogLS1iYXRjaC1zaXplIDE2CiIiIgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGpzb24KaW1wb3J0IGxvZ2dpbmcKaW1wb3J0IHNodXRpbApmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgdG9yY2gKaW1wb3J0IHRvcmNoLm5uIGFzIG5uCmZyb20gc2tsZWFybi5tZXRyaWNzIGltcG9ydCBjbGFzc2lmaWNhdGlvbl9yZXBvcnQsIGYxX3Njb3JlCmZyb20gc2tsZWFybi5tb2RlbF9zZWxlY3Rpb24gaW1wb3J0IHRyYWluX3Rlc3Rfc3BsaXQKZnJvbSB0b3JjaC51dGlscy5kYXRhIGltcG9ydCBEYXRhTG9hZGVyLCBEYXRhc2V0CmZyb20gdHFkbSBpbXBvcnQgdHFkbQpmcm9tIHRyYW5zZm9ybWVycyBpbXBvcnQgKAogICAgQXV0b0NvbmZpZywKICAgIEF1dG9Nb2RlbEZvclNlcXVlbmNlQ2xhc3NpZmljYXRpb24sCiAgICBBdXRvVG9rZW5pemVyLAogICAgZ2V0X2xpbmVhcl9zY2hlZHVsZV93aXRoX3dhcm11cCwKKQoKbG9nZ2luZy5iYXNpY0NvbmZpZyhsZXZlbD1sb2dnaW5nLklORk8pCmxvZ2dlciA9IGxvZ2dpbmcuZ2V0TG9nZ2VyKF9fbmFtZV9fKQoKREVWSUNFID0gdG9yY2guZGV2aWNlKCJjdWRhIiBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpIGVsc2UgImNwdSIpCkZPUk1BTF9DTEFTU0VTID0gewogICAgImFmZmlybWluZ19jb25zZXF1ZW50IiwKICAgICJkZW55aW5nX2FudGVjZWRlbnQiLAogICAgInVuZGlzdHJpYnV0ZWRfbWlkZGxlIiwKICAgICJpbGxpY2l0X21ham9yIiwKICAgICJpbGxpY2l0X21pbm9yIiwKICAgICJleGNsdXNpdmVfcHJlbWlzZXMiLAogICAgImV4aXN0ZW50aWFsX2ZhbGxhY3kiLAp9CgoKY2xhc3MgRmFsbGFjeURhdGFzZXQoRGF0YXNldCk6CiAgICBkZWYgX19pbml0X18oc2VsZiwgdGV4dHMsIGxhYmVscywgdG9rZW5pemVyLCBtYXhfbGVuZ3RoPTI1Nik6CiAgICAgICAgc2VsZi50ZXh0cyA9IHRleHRzCiAgICAgICAgc2VsZi5sYWJlbHMgPSBsYWJlbHMKICAgICAgICBzZWxmLnRva2VuaXplciA9IHRva2VuaXplcgogICAgICAgIHNlbGYubWF4X2xlbmd0aCA9IG1heF9sZW5ndGgKCiAgICBkZWYgX19sZW5fXyhzZWxmKToKICAgICAgICByZXR1cm4gbGVuKHNlbGYudGV4dHMpCgogICAgZGVmIF9fZ2V0aXRlbV9fKHNlbGYsIGlkeCk6CiAgICAgICAgZW5jID0gc2VsZi50b2tlbml6ZXIoCiAgICAgICAgICAgIHNlbGYudGV4dHNbaWR4XSwKICAgICAgICAgICAgdHJ1bmNhdGlvbj1UcnVlLAogICAgICAgICAgICBwYWRkaW5nPSJtYXhfbGVuZ3RoIiwKICAgICAgICAgICAgbWF4X2xlbmd0aD1zZWxmLm1heF9sZW5ndGgsCiAgICAgICAgICAgIHJldHVybl90ZW5zb3JzPSJwdCIsCiAgICAgICAgKQogICAgICAgIHJldHVybiB7CiAgICAgICAgICAgICJpbnB1dF9pZHMiOiBlbmNbImlucHV0X2lkcyJdLnNxdWVlemUoMCksCiAgICAgICAgICAgICJhdHRlbnRpb25fbWFzayI6IGVuY1siYXR0ZW50aW9uX21hc2siXS5zcXVlZXplKDApLAogICAgICAgICAgICAibGFiZWxzIjogdG9yY2gudGVuc29yKHNlbGYubGFiZWxzW2lkeF0sIGR0eXBlPXRvcmNoLmxvbmcpLAogICAgICAgIH0KCgpkZWYgbG9hZF9kYXRhKHBhdGg6IFBhdGgpOgogICAgIiIiTG9hZCB2MS4zIEpTT04sIGtlZXAgb25seSBzYW1wbGVzIHdpdGggYSBgZmFsbGFjeWAgbGFiZWwuIiIiCiAgICB3aXRoIG9wZW4ocGF0aCkgYXMgZjoKICAgICAgICBkYXRhID0ganNvbi5sb2FkKGYpCgogICAgbGFiZWxlZCA9IFtkIGZvciBkIGluIGRhdGEgaWYgImZhbGxhY3kiIGluIGRdCiAgICBkcm9wcGVkID0gbGVuKGRhdGEpIC0gbGVuKGxhYmVsZWQpCiAgICBpZiBkcm9wcGVkOgogICAgICAgIGxvZ2dlci5pbmZvKGYiRHJvcHBlZCB7ZHJvcHBlZH0gdW5sYWJlbGVkIG5lYXItbWlzcyBzYW1wbGVzIChTdGFnZSAxIGRhdGEpLiIpCgogICAgdGV4dHMgPSBbZFsidGV4dCJdIGZvciBkIGluIGxhYmVsZWRdCiAgICBsYWJlbF9saXN0ID0gc29ydGVkKHNldChkWyJmYWxsYWN5Il0gZm9yIGQgaW4gbGFiZWxlZCkpCiAgICBsYWJlbDJpZCA9IHtsOiBpIGZvciBpLCBsIGluIGVudW1lcmF0ZShsYWJlbF9saXN0KX0KICAgIGxhYmVscyA9IFtsYWJlbDJpZFtkWyJmYWxsYWN5Il1dIGZvciBkIGluIGxhYmVsZWRdCiAgICByZXR1cm4gdGV4dHMsIGxhYmVscywgbGFiZWxfbGlzdAoKCmRlZiBjb21wdXRlX2NsYXNzX3dlaWdodHMobGFiZWxzOiBsaXN0W2ludF0sIG51bV9sYWJlbHM6IGludCkgLT4gdG9yY2guVGVuc29yOgogICAgY291bnRzID0gbnAuYmluY291bnQobGFiZWxzLCBtaW5sZW5ndGg9bnVtX2xhYmVscykKICAgIHdlaWdodHMgPSAxLjAgLyAobnAuc3FydChjb3VudHMpICsgMWUtNikKICAgIHdlaWdodHMgPSB3ZWlnaHRzIC8gd2VpZ2h0cy5zdW0oKSAqIG51bV9sYWJlbHMKICAgIGxvZ2dlci5pbmZvKGYiQ2xhc3MgaW1iYWxhbmNlIHJhdGlvIChtYXgvbWluIGNvdW50KToge2NvdW50cy5tYXgoKSAvIGNvdW50cy5taW4oKTouMWZ9OjEiKQogICAgcmV0dXJuIHRvcmNoLnRlbnNvcih3ZWlnaHRzLCBkdHlwZT10b3JjaC5mbG9hdCkudG8oREVWSUNFKQoKCmRlZiB0cmFpbihhcmdzKToKICAgIGxvZ2dlci5pbmZvKGYiVXNpbmcgZGV2aWNlOiB7REVWSUNFfSAoe3RvcmNoLmN1ZGEuZ2V0X2RldmljZV9uYW1lKDApIGlmIERFVklDRS50eXBlID09ICdjdWRhJyBlbHNlICdjcHUnfSkiKQoKICAgIGRhdGFfcGF0aCA9IFBhdGgoYXJncy5kYXRhKQogICAgaWYgbm90IGRhdGFfcGF0aC5leGlzdHMoKToKICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcihmIlRyYWluaW5nIGRhdGEgbm90IGZvdW5kIGF0IHtkYXRhX3BhdGh9IikKICAgIG91dHB1dF9kaXIgPSBQYXRoKGFyZ3Mub3V0cHV0X2RpcikKICAgIG91dHB1dF9kaXIubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQoKICAgIHRleHRzLCBsYWJlbHMsIGxhYmVsX2xpc3QgPSBsb2FkX2RhdGEoZGF0YV9wYXRoKQogICAgbnVtX2xhYmVscyA9IGxlbihsYWJlbF9saXN0KQogICAgbG9nZ2VyLmluZm8oZiJMb2FkZWQge2xlbih0ZXh0cyl9IHNhbXBsZXMgYWNyb3NzIHtudW1fbGFiZWxzfSBjbGFzc2VzLiIpCgogICAgbnAucmFuZG9tLnNlZWQoYXJncy5zZWVkKQogICAgdG9yY2gubWFudWFsX3NlZWQoYXJncy5zZWVkKQogICAgWF90cmFpbiwgWF92YWwsIHlfdHJhaW4sIHlfdmFsID0gdHJhaW5fdGVzdF9zcGxpdCgKICAgICAgICB0ZXh0cywgbGFiZWxzLCB0ZXN0X3NpemU9YXJncy52YWxfc3BsaXQsIHJhbmRvbV9zdGF0ZT1hcmdzLnNlZWQsIHN0cmF0aWZ5PWxhYmVscwogICAgKQogICAgbG9nZ2VyLmluZm8oZiJUcmFpbjoge2xlbihYX3RyYWluKX0gLyBWYWw6IHtsZW4oWF92YWwpfSIpCgogICAgbW9kZWxfbmFtZSA9IGFyZ3MuYmFzZV9tb2RlbAogICAgdG9rZW5pemVyID0gQXV0b1Rva2VuaXplci5mcm9tX3ByZXRyYWluZWQobW9kZWxfbmFtZSkKICAgIHRyYWluX2RzID0gRmFsbGFjeURhdGFzZXQoWF90cmFpbiwgeV90cmFpbiwgdG9rZW5pemVyLCBtYXhfbGVuZ3RoPWFyZ3MubWF4X2xlbmd0aCkKICAgIHZhbF9kcyA9IEZhbGxhY3lEYXRhc2V0KFhfdmFsLCB5X3ZhbCwgdG9rZW5pemVyLCBtYXhfbGVuZ3RoPWFyZ3MubWF4X2xlbmd0aCkKCiAgICBuX2dwdXMgPSB0b3JjaC5jdWRhLmRldmljZV9jb3VudCgpIGlmIERFVklDRS50eXBlID09ICJjdWRhIiBlbHNlIDAKICAgIGxvYWRlcl9iYXRjaCA9IGFyZ3MuYmF0Y2hfc2l6ZSAqIG5fZ3B1cyBpZiBuX2dwdXMgPiAxIGVsc2UgYXJncy5iYXRjaF9zaXplCiAgICB0cmFpbl9sb2FkZXIgPSBEYXRhTG9hZGVyKHRyYWluX2RzLCBiYXRjaF9zaXplPWxvYWRlcl9iYXRjaCwgc2h1ZmZsZT1UcnVlKQogICAgdmFsX2xvYWRlciA9IERhdGFMb2FkZXIodmFsX2RzLCBiYXRjaF9zaXplPWxvYWRlcl9iYXRjaCAqIDIpCgogICAgY2xhc3Nfd2VpZ2h0cyA9IGNvbXB1dGVfY2xhc3Nfd2VpZ2h0cyh5X3RyYWluLCBudW1fbGFiZWxzKQoKICAgIGNvbmZpZyA9IEF1dG9Db25maWcuZnJvbV9wcmV0cmFpbmVkKG1vZGVsX25hbWUpCiAgICBjb25maWcubnVtX2xhYmVscyA9IG51bV9sYWJlbHMKICAgIGNvbmZpZy5pZDJsYWJlbCA9IHtpOiBsIGZvciBpLCBsIGluIGVudW1lcmF0ZShsYWJlbF9saXN0KX0KICAgIGNvbmZpZy5sYWJlbDJpZCA9IHtsOiBpIGZvciBpLCBsIGluIGVudW1lcmF0ZShsYWJlbF9saXN0KX0KICAgIGNvbmZpZy5vdXRwdXRfaGlkZGVuX3N0YXRlcyA9IFRydWUKCiAgICBtb2RlbCA9IEF1dG9Nb2RlbEZvclNlcXVlbmNlQ2xhc3NpZmljYXRpb24uZnJvbV9wcmV0cmFpbmVkKAogICAgICAgIG1vZGVsX25hbWUsIGNvbmZpZz1jb25maWcsIHRvcmNoX2R0eXBlPXRvcmNoLmZsb2F0MzIsIGlnbm9yZV9taXNtYXRjaGVkX3NpemVzPVRydWUKICAgICkudG8oREVWSUNFKQogICAgbG9nZ2VyLmluZm8oZiJNb2RlbCBwYXJhbXM6IHtzdW0ocC5udW1lbCgpIGZvciBwIGluIG1vZGVsLnBhcmFtZXRlcnMoKSk6LH0iKQoKICAgIG5fZ3B1cyA9IHRvcmNoLmN1ZGEuZGV2aWNlX2NvdW50KCkgaWYgREVWSUNFLnR5cGUgPT0gImN1ZGEiIGVsc2UgMAogICAgdXNlX2RwID0gbl9ncHVzID4gMQogICAgaWYgdXNlX2RwOgogICAgICAgIHRvcmNoLmJhY2tlbmRzLmN1ZG5uLmJlbmNobWFyayA9IFRydWUKICAgICAgICBtb2RlbCA9IG5uLkRhdGFQYXJhbGxlbChtb2RlbCkKICAgICAgICBsb2dnZXIuaW5mbygKICAgICAgICAgICAgZiJEYXRhUGFyYWxsZWwgYWN0aXZlIGFjcm9zcyB7bl9ncHVzfSBHUFVzOyBlZmZlY3RpdmUgcGVyLUdQVSBiYXRjaCA9IHthcmdzLmJhdGNoX3NpemV9IgogICAgICAgICkKCiAgICBvcHRpbWl6ZXIgPSB0b3JjaC5vcHRpbS5BZGFtVyhtb2RlbC5wYXJhbWV0ZXJzKCksIGxyPWFyZ3MubHIsIHdlaWdodF9kZWNheT0wLjAxKQogICAgdG90YWxfc3RlcHMgPSBsZW4odHJhaW5fbG9hZGVyKSAqIGFyZ3MuZXBvY2hzCiAgICBzY2hlZHVsZXIgPSBnZXRfbGluZWFyX3NjaGVkdWxlX3dpdGhfd2FybXVwKAogICAgICAgIG9wdGltaXplciwgbnVtX3dhcm11cF9zdGVwcz10b3RhbF9zdGVwcyAvLyAxMCwgbnVtX3RyYWluaW5nX3N0ZXBzPXRvdGFsX3N0ZXBzCiAgICApCiAgICBzY2FsZXIgPSB0b3JjaC5hbXAuR3JhZFNjYWxlcigiY3VkYSIpIGlmIERFVklDRS50eXBlID09ICJjdWRhIiBlbHNlIE5vbmUKICAgIGxvc3NfZm4gPSBubi5Dcm9zc0VudHJvcHlMb3NzKHdlaWdodD1jbGFzc193ZWlnaHRzKQoKICAgIGJlc3RfZjEgPSAwLjAKCiAgICBmb3IgZXBvY2ggaW4gcmFuZ2UoYXJncy5lcG9jaHMpOgogICAgICAgIG1vZGVsLnRyYWluKCkKICAgICAgICB0cmFpbl9sb3NzID0gMC4wCiAgICAgICAgcGJhciA9IHRxZG0odHJhaW5fbG9hZGVyLCBkZXNjPWYiRXBvY2gge2Vwb2NoICsgMX0ve2FyZ3MuZXBvY2hzfSIpCgogICAgICAgIGZvciBiYXRjaCBpbiBwYmFyOgogICAgICAgICAgICBvcHRpbWl6ZXIuemVyb19ncmFkKCkKICAgICAgICAgICAgaW5wdXRfaWRzID0gYmF0Y2hbImlucHV0X2lkcyJdLnRvKERFVklDRSkKICAgICAgICAgICAgYXR0ZW50aW9uX21hc2sgPSBiYXRjaFsiYXR0ZW50aW9uX21hc2siXS50byhERVZJQ0UpCiAgICAgICAgICAgIGxhYmVsc19iYXRjaCA9IGJhdGNoWyJsYWJlbHMiXS50byhERVZJQ0UpCgogICAgICAgICAgICBpZiBzY2FsZXI6CiAgICAgICAgICAgICAgICB3aXRoIHRvcmNoLmFtcC5hdXRvY2FzdChkZXZpY2VfdHlwZT0iY3VkYSIpOgogICAgICAgICAgICAgICAgICAgIG91dHB1dHMgPSBtb2RlbChpbnB1dF9pZHMsIGF0dGVudGlvbl9tYXNrPWF0dGVudGlvbl9tYXNrKQogICAgICAgICAgICAgICAgICAgIGxvc3MgPSBsb3NzX2ZuKG91dHB1dHMubG9naXRzLCBsYWJlbHNfYmF0Y2gpCiAgICAgICAgICAgICAgICBzY2FsZXIuc2NhbGUobG9zcykuYmFja3dhcmQoKQogICAgICAgICAgICAgICAgc2NhbGVyLnVuc2NhbGVfKG9wdGltaXplcikKICAgICAgICAgICAgICAgIHRvcmNoLm5uLnV0aWxzLmNsaXBfZ3JhZF9ub3JtXyhtb2RlbC5wYXJhbWV0ZXJzKCksIDEuMCkKICAgICAgICAgICAgICAgIHNjYWxlci5zdGVwKG9wdGltaXplcikKICAgICAgICAgICAgICAgIHNjYWxlci51cGRhdGUoKQogICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgb3V0cHV0cyA9IG1vZGVsKGlucHV0X2lkcywgYXR0ZW50aW9uX21hc2s9YXR0ZW50aW9uX21hc2spCiAgICAgICAgICAgICAgICBsb3NzID0gbG9zc19mbihvdXRwdXRzLmxvZ2l0cywgbGFiZWxzX2JhdGNoKQogICAgICAgICAgICAgICAgbG9zcy5iYWNrd2FyZCgpCiAgICAgICAgICAgICAgICB0b3JjaC5ubi51dGlscy5jbGlwX2dyYWRfbm9ybV8obW9kZWwucGFyYW1ldGVycygpLCAxLjApCiAgICAgICAgICAgICAgICBvcHRpbWl6ZXIuc3RlcCgpCgogICAgICAgICAgICBzY2hlZHVsZXIuc3RlcCgpCiAgICAgICAgICAgIHRyYWluX2xvc3MgKz0gbG9zcy5pdGVtKCkKICAgICAgICAgICAgcGJhci5zZXRfcG9zdGZpeCh7Imxvc3MiOiBmIntsb3NzLml0ZW0oKTouNGZ9In0pCgogICAgICAgIG1vZGVsLmV2YWwoKQogICAgICAgIGFsbF9wcmVkcywgYWxsX2xhYmVscyA9IFtdLCBbXQogICAgICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgICAgICBmb3IgYmF0Y2ggaW4gdmFsX2xvYWRlcjoKICAgICAgICAgICAgICAgIG91dHB1dHMgPSBtb2RlbCgKICAgICAgICAgICAgICAgICAgICBiYXRjaFsiaW5wdXRfaWRzIl0udG8oREVWSUNFKSwKICAgICAgICAgICAgICAgICAgICBhdHRlbnRpb25fbWFzaz1iYXRjaFsiYXR0ZW50aW9uX21hc2siXS50byhERVZJQ0UpLAogICAgICAgICAgICAgICAgKQogICAgICAgICAgICAgICAgcHJlZHMgPSB0b3JjaC5hcmdtYXgob3V0cHV0cy5sb2dpdHMsIGRpbT0xKQogICAgICAgICAgICAgICAgYWxsX3ByZWRzLmV4dGVuZChwcmVkcy5jcHUoKS5udW1weSgpKQogICAgICAgICAgICAgICAgYWxsX2xhYmVscy5leHRlbmQoYmF0Y2hbImxhYmVscyJdLm51bXB5KCkpCgogICAgICAgIGYxID0gZjFfc2NvcmUoYWxsX2xhYmVscywgYWxsX3ByZWRzLCBhdmVyYWdlPSJtYWNybyIpCiAgICAgICAgbG9nZ2VyLmluZm8oZiJcbkVwb2NoIHtlcG9jaCArIDF9OiBsb3NzPXt0cmFpbl9sb3NzIC8gbGVuKHRyYWluX2xvYWRlcik6LjRmfSwgdmFsX21hY3JvX2YxPXtmMTouNGZ9IikKICAgICAgICBwcmludChjbGFzc2lmaWNhdGlvbl9yZXBvcnQoYWxsX2xhYmVscywgYWxsX3ByZWRzLCB0YXJnZXRfbmFtZXM9bGFiZWxfbGlzdCwgemVyb19kaXZpc2lvbj0wLCBkaWdpdHM9MykpCgogICAgICAgIGZvcm1hbCA9IFtsIGZvciBsIGluIGxhYmVsX2xpc3QgaWYgbCBpbiBGT1JNQUxfQ0xBU1NFU10KICAgICAgICBpZiBmb3JtYWw6CiAgICAgICAgICAgIGZvcm1hbF9mMXMgPSB7CiAgICAgICAgICAgICAgICBsOiBmMV9zY29yZSgKICAgICAgICAgICAgICAgICAgICBbMSBpZiB5ID09IGxhYmVsX2xpc3QuaW5kZXgobCkgZWxzZSAwIGZvciB5IGluIGFsbF9sYWJlbHNdLAogICAgICAgICAgICAgICAgICAgIFsxIGlmIHAgPT0gbGFiZWxfbGlzdC5pbmRleChsKSBlbHNlIDAgZm9yIHAgaW4gYWxsX3ByZWRzXSwKICAgICAgICAgICAgICAgICAgICB6ZXJvX2RpdmlzaW9uPTAsCiAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICAgICBmb3IgbCBpbiBmb3JtYWwKICAgICAgICAgICAgfQogICAgICAgICAgICBsb2dnZXIuaW5mbyhmIkZvcm1hbC1jbGFzcyBGMToge2Zvcm1hbF9mMXN9IikKCiAgICAgICAgaWYgZjEgPiBiZXN0X2YxOgogICAgICAgICAgICBiZXN0X2YxID0gZjEKICAgICAgICAgICAgbG9nZ2VyLmluZm8oZiJOZXcgYmVzdCBGMToge2YxOi40Zn0uIFNhdmluZyBtb2RlbC4uLiIpCiAgICAgICAgICAgIHNhdmVfbW9kZWwgPSBtb2RlbC5tb2R1bGUgaWYgaXNpbnN0YW5jZShtb2RlbCwgbm4uRGF0YVBhcmFsbGVsKSBlbHNlIG1vZGVsCiAgICAgICAgICAgIHNhdmVfbW9kZWwuc2F2ZV9wcmV0cmFpbmVkKG91dHB1dF9kaXIpCiAgICAgICAgICAgIHRva2VuaXplci5zYXZlX3ByZXRyYWluZWQob3V0cHV0X2RpcikKICAgICAgICAgICAgd2l0aCBvcGVuKG91dHB1dF9kaXIgLyAibGFiZWxzLmpzb24iLCAidyIpIGFzIGY6CiAgICAgICAgICAgICAgICBqc29uLmR1bXAobGFiZWxfbGlzdCwgZiwgaW5kZW50PTIpCgogICAgbG9nZ2VyLmluZm8oZiJUcmFpbmluZyBjb21wbGV0ZS4gQmVzdCBtYWNybyBGMToge2Jlc3RfZjE6LjRmfSIpCgogICAgbW9kZWxfc2l6ZSA9IHN1bShmLnN0YXQoKS5zdF9zaXplIGZvciBmIGluIG91dHB1dF9kaXIuZ2xvYigiKiIpIGlmIGYuaXNfZmlsZSgpKQogICAgbG9nZ2VyLmluZm8oZiJNb2RlbCBmb2xkZXIgc2l6ZToge21vZGVsX3NpemUgLyAoMTAyNCoqMik6LjJmfSBNQiIpCiAgICBpZiBhcmdzLmV4cG9ydF9vbm54OgogICAgICAgIHRyeToKICAgICAgICAgICAgZXhwb3J0X29ubngoc2F2ZV9tb2RlbCwgdG9rZW5pemVyLCBvdXRwdXRfZGlyLCBsYWJlbF9saXN0KQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZTogICMgcHJhZ21hOiBubyBjb3ZlciAtIGV4cG9ydCBpcyBiZXN0LWVmZm9ydAogICAgICAgICAgICBsb2dnZXIud2FybmluZyhmIk9OTlggZXhwb3J0IGZhaWxlZCAobm9uLWZhdGFsKToge2V9IikKCiAgICByZXR1cm4gYmVzdF9mMQoKCmRlZiBleHBvcnRfb25ueChtb2RlbCwgdG9rZW5pemVyLCBvdXRwdXRfZGlyLCBsYWJlbF9saXN0KToKICAgIGxvZ2dlci5pbmZvKCJFeHBvcnRpbmcgdG8gT05OWC4uLiIpCiAgICBtb2RlbC5ldmFsKCkKICAgIGRldmljZSA9IG5leHQobW9kZWwucGFyYW1ldGVycygpKS5kZXZpY2UKCiAgICBkdW1teSA9IHRva2VuaXplcigKICAgICAgICAidGVzdCBpbnB1dCIsIHJldHVybl90ZW5zb3JzPSJwdCIsIHBhZGRpbmc9Im1heF9sZW5ndGgiLCB0cnVuY2F0aW9uPVRydWUsIG1heF9sZW5ndGg9MjU2CiAgICApCiAgICBkdW1teV9pZHMgPSBkdW1teVsiaW5wdXRfaWRzIl0udG8oZGV2aWNlKQogICAgZHVtbXlfbWFzayA9IGR1bW15WyJhdHRlbnRpb25fbWFzayJdLnRvKGRldmljZSkKCiAgICBvbm54X3BhdGggPSBvdXRwdXRfZGlyIC8gIm1vZGVsLm9ubngiCiAgICB0b3JjaC5vbm54LmV4cG9ydCgKICAgICAgICBtb2RlbCwKICAgICAgICAoZHVtbXlfaWRzLCBkdW1teV9tYXNrKSwKICAgICAgICBvbm54X3BhdGgsCiAgICAgICAgb3BzZXRfdmVyc2lvbj0xNCwKICAgICAgICBpbnB1dF9uYW1lcz1bImlucHV0X2lkcyIsICJhdHRlbnRpb25fbWFzayJdLAogICAgICAgIG91dHB1dF9uYW1lcz1bImxvZ2l0cyJdLAogICAgICAgIGR5bmFtaWNfYXhlcz17CiAgICAgICAgICAgICJpbnB1dF9pZHMiOiB7MDogImJhdGNoIiwgMTogInNlcXVlbmNlIn0sCiAgICAgICAgICAgICJhdHRlbnRpb25fbWFzayI6IHswOiAiYmF0Y2giLCAxOiAic2VxdWVuY2UifSwKICAgICAgICAgICAgImxvZ2l0cyI6IHswOiAiYmF0Y2gifSwKICAgICAgICB9LAogICAgKQogICAgbG9nZ2VyLmluZm8oZiJPTk5YIGV4cG9ydCBjb21wbGV0ZToge29ubnhfcGF0aC5uYW1lfSAoe29ubnhfcGF0aC5zdGF0KCkuc3Rfc2l6ZSAvICgxMDI0KioyKTouMWZ9IE1CKSIpCiAgICB3aXRoIG9wZW4ob3V0cHV0X2RpciAvICJtb2RlbF9jb25maWdfb25ueC5qc29uIiwgInciKSBhcyBmOgogICAgICAgIGpzb24uZHVtcCh7ImZpbmVfbGFiZWxzIjogbGFiZWxfbGlzdCwgIm51bV9sYWJlbHMiOiBsZW4obGFiZWxfbGlzdCl9LCBmLCBpbmRlbnQ9MikKCiAgICB6aXBfcGF0aCA9IHNodXRpbC5tYWtlX2FyY2hpdmUoc3RyKG91dHB1dF9kaXIucGFyZW50IC8gb3V0cHV0X2Rpci5uYW1lKSwgInppcCIsIG91dHB1dF9kaXIpCiAgICBsb2dnZXIuaW5mbyhmIlBhY2thZ2UgcmVhZHk6IHt6aXBfcGF0aH0iKQo=").decode()
_NS3 = {}
exec(_STAGE3_SRC, _NS3)
print("stage 3 code loaded:", _NS3["train"].__name__)


In [ ]:
import json
from pathlib import Path

Path("cloud_training/results").mkdir(parents=True, exist_ok=True)
results = {"data": "data/unified_training_data_v1.3.json", "seeds": {}, "status": "running"}

for seed in SEEDS_STAGE3:
    args = argparse.Namespace(
        data="data/unified_training_data_v1.3.json",
        output_dir=f"models/seed_variation/seed_{seed}",
        base_model="microsoft/deberta-v3-small",
        epochs=6, batch_size=BATCH_STAGE3, lr=1e-5,
        seed=seed, max_length=256, val_split=0.1, export_onnx=False,
    )
    print(f"===== SEED {seed} =====")
    try:
        f1 = _NS3["train"](args)
        results["seeds"][str(seed)] = round(float(f1), 4)
        print(f"Seed {seed}: best val macro F1 = {f1:.4f}")
    except Exception as e:
        results["seeds"][str(seed)] = f"ERROR: {e}"
        print(f"Seed {seed} FAILED: {e}")
    # incremental write so a partial run still leaves a results file
    with open("cloud_training/results/seed_variation.json", "w") as f:
        json.dump(results, f, indent=2)

f1s = [v for v in results["seeds"].values() if isinstance(v, (int, float))]
if len(f1s) == len(SEEDS_STAGE3):
    import statistics
    results["macro_f1_mean"] = round(statistics.mean(f1s), 4)
    results["macro_f1_std"] = round(statistics.stdev(f1s), 4)
    results["status"] = "complete"
    print(f"\nMean macro F1: {results['macro_f1_mean']:.4f} +/- {results['macro_f1_std']:.4f}")
else:
    results["status"] = "partial"
    print("\nSome seeds failed - partial results saved. Check the errors above.")

with open("cloud_training/results/seed_variation.json", "w") as f:
    json.dump(results, f, indent=2)
print("Saved cloud_training/results/seed_variation.json")


## 4. Results & download


In [ ]:
import json, os, zipfile

print("--- seed variation results ---")
if os.path.exists("cloud_training/results/seed_variation.json"):
    with open("cloud_training/results/seed_variation.json") as f:
        print(json.dumps(json.load(f), indent=2))
else:
    print("!! seed_variation.json missing - the seed-variation cell above did not complete.")

if not os.path.exists("models/stage1_v13_classifier.zip") and os.path.isdir("models/stage1_v13_classifier"):
    with zipfile.ZipFile("models/stage1_v13_classifier.zip", "w", zipfile.ZIP_DEFLATED) as z:
        for root, _, files in os.walk("models/stage1_v13_classifier"):
            for fn in files:
                fp = os.path.join(root, fn)
                z.write(fp, os.path.relpath(fp, "models"))
    print("\nZipped models/stage1_v13_classifier.zip")

print("\nOutputs ready for download:")
for f in ["models/stage1_v13_classifier.zip", "cloud_training/results/seed_variation.json"]:
    print(" -", f, f"({os.path.getsize(f)/1e6:.1f} MB)" if os.path.exists(f) else "(MISSING)")
